## 머신러닝 모델링 및 Flask Web Serving 구현

### 1. 평가 개요
- 과제명: 데이터 전처리 및 하이퍼파라미터 최적화 기반 이직 예측 웹 서비스 구현
- 제공 데이터: employee_data.csv (1,000행)
- 데이터셋 주요 컬럼 구조:
    - Age (수치형): 직원의 나이 (결측치 존재)
    - Gender (범주형): Male, Female
    - OverTime (범주형): Yes, No
    - MonthlyIncome (수치형): 월급 (단위: 만원)
    - Attrition (Target, 범주형): 이직여부 (1: 이직함, 0: 잔류) — 예측해야 할 종속변수

### Ⅰ. 시험 지시서
#### [작업 1] Jupyter Notebook 데이터 분석 및 모델링
- 데이터 로드 및 결측치 처리:
    - employee_data.csv 파일을 읽어온 후 결측치를 확인하시오.
    - 수치형 변수(Age)의 결측치는 전체 데이터의 중앙값(Median)으로 대체하시오.
- 데이터 인코딩 및 스케일링:
    - 문자열 범주형 변수(Gender, OverTime)를 수치형 변수(0 또는 1)로 변환(인코딩)하시오.
    - 수치형 독립변수 데이터에 대해 표준화 스케일러(StandardScaler)를 적용하시오.
- 데이터셋 분리:
    - 독립변수(\(X\))와 종속변수(\(y\), Attrition)를 분리하고, 학습 데이터와 테스트 데이터를 8:2 비율로 분리하시오. (random_state=42 고정)
- 모델 파라미터 최적화 및 저장:
    - GridSearchCV를 활용하여 알고리즘(예: DecisionTree, RandomForest, SVM 등 자유 선택)의 최적 하이퍼파라미터를 도출하시오. (5-fold 교차검증 적용)
    - 최적화된 최종 모델을 joblib 라이브러리를 사용하여 best_model.pkl 파일로 저장하시오.
#### [작업 2] Flask 기반 웹 추론 서비스 구현
- 사용자가 웹 화면에서 Age, Gender, OverTime, MonthlyIncome을 직접 입력할 수 있는 HTML 폼 페이지(index.html)를 제작하시오.
- 사용자가 제출(Submit) 버튼을 누르면 /predict 엔드포인트로 데이터가 POST 방식으로 전달되도록 구현하시오.
- Flask 서버(app.py)는 전달받은 데이터를 학습 당시 사용했던 전처리 기준과 동일하게 변환한 후, 저장된 best_model.pkl 파일을 로드하여 이직 여부(0 또는 1)를 추론하도록 하시오.
- 추론 결과에 따라 화면에 "이직 확률이 높습니다." 또는 "잔류할 가능성이 높습니다."를 출력하시오.


In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

data = pd.read_csv('employee_data.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Age            950 non-null    float64
 1   Gender         1000 non-null   str    
 2   OverTime       1000 non-null   str    
 3   MonthlyIncome  1000 non-null   float64
 4   Attrition      1000 non-null   int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 39.2 KB


In [43]:
data['Age'] = data['Age'].fillna(data['Age'].median())
data.isna().sum()

Age              0
Gender           0
OverTime         0
MonthlyIncome    0
Attrition        0
dtype: int64

In [44]:
data['Gender'].unique(), data['OverTime'].unique()

(<StringArray>
 ['Male', 'Female']
 Length: 2, dtype: str,
 <StringArray>
 ['No', 'Yes']
 Length: 2, dtype: str)

In [45]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
data['Gender'] = enc.fit_transform(data['Gender'])
enc_gen = enc.classes_
data['OverTime'] = enc.fit_transform(data['OverTime'])
enc_OT = enc.classes_

In [46]:
data.head(10), enc_gen, enc_OT

(    Age  Gender  OverTime  MonthlyIncome  Attrition
 0  50.0       1         0          636.0          0
 1  36.0       0         0          391.0          0
 2  29.0       1         1          356.0          1
 3  42.0       0         0          656.0          0
 4  40.0       1         0          470.0          0
 5  42.0       1         0          582.0          0
 6  32.0       0         1          466.0          1
 7  32.0       1         1          455.0          1
 8  45.0       1         1          682.0          0
 9  57.0       1         0          778.0          0,
 array(['Female', 'Male'], dtype=object),
 array(['No', 'Yes'], dtype=object))

In [47]:
from sklearn.preprocessing import StandardScaler

age_scaler = StandardScaler()
income_scaler = StandardScaler()

data['Age'] = age_scaler.fit_transform(data[['Age']])
data['MonthlyIncome'] = income_scaler.fit_transform(data[['MonthlyIncome']])
data.describe()

,Age,Gender,OverTime,MonthlyIncome,Attrition
count,1.000000e+03,1000.000000,1000.000000,1.000000e+03,1000.000000
mean,-2.344791e-16,0.604000,0.334000,2.327027e-16,0.250000
std,1.000500e+00,0.489309,0.471876,1.000500e+00,0.433229
min,-1.754085e+00,0.000000,0.000000,-2.297985e+00,0.000000
25%,-8.340784e-01,0.000000,0.000000,-7.860832e-01,0.000000
50%,8.592866e-02,1.000000,0.000000,5.899718e-02,0.000000
75%,8.219343e-01,1.000000,1.000000,7.538778e-01,0.250000
max,1.649941e+00,1.000000,1.000000,2.191505e+00,1.000000


In [48]:
import joblib

joblib.dump(age_scaler, 'age.scaler')
joblib.dump(income_scaler, 'MonthlyIncome.scaler')

['MonthlyIncome.scaler']

In [49]:
X = data.drop(columns=['Attrition'])
y = data['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 4), (200, 4), (800,), (200,))

In [50]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn import set_config
set_config(display='text')

params = {'C': [0.001, 0.01, 0.1, 1, 10],
          'solver' : ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'max_iter': [1000]}
grid_search = GridSearchCV(LogisticRegression(), params, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10], 'max_iter': [1000],
                         'solver': ['lbfgs', 'newton-cg', 'newton-cholesky',
                                    'sag', 'saga']})

In [51]:
print(grid_search.best_params_)
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
joblib.dump(best_model, 'best_model.pkl')

{'C': 0.1, 'max_iter': 1000, 'solver': 'lbfgs'}
0.8175000000000001


['best_model.pkl']

In [52]:
pred = best_model.predict(X_test)
print(pred)
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 1 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0
 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0]
[[138   9]
 [ 32  21]]
              precision    recall  f1-score   support

           0       0.81      0.94      0.87       147
           1       0.70      0.40      0.51        53

    accuracy                           0.80       200
   macro avg       0.76      0.67      0.69       200
weighted avg       0.78      0.80      0.77       200

